# 02 — Data Preparation

This notebook generates and audits the reproducible article-week dataset used by all three learning tasks. The pipeline validates raw H&M sources, aggregates complete weeks, creates leakage-safe historical features, defines next-week regression and classification targets, assigns chronological partitions, and publishes an article-level clustering table.

The supervised candidate population contains articles with at least one recorded sale in the current week. Targets represent recorded sales rather than latent demand because stock availability is unknown.

## 1. Generate or reuse the validated publication

A missing publication is generated automatically. Set `REBUILD_PIPELINE = True` to create a fresh versioned run after changing source data, feature definitions, or configuration. The pipeline never overwrites raw files, and an incomplete run cannot replace `CURRENT.json`.

🔴 **Data update for notebook 07 (Iratxe, 27 October 2026).** I ran `python -B src/data/process_data.py` again. `CURRENT.json` now selects run `20261027T145552-5a1ac59a`; the earlier run is still saved. This update supplies `target_high_demand` and the historical columns needed by notebook 07. High demand means at least **16 next-week sales**, calculated from training data. The **more than 20 sales** groups in notebook 07 are a separate error analysis. The saved outputs below were recalculated with this run.

**Code walkthrough.** The next cell finds the project folder. With `REBUILD_PIPELINE = False`, it reuses the newly selected run instead of processing the 31,8 million transactions again.


In [1]:
from pathlib import Path
import hashlib
import json
import subprocess
import sys

import numpy as np
import pandas as pd

ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "src/data/process_data.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from the fashion-demand-ml project.")
sys.path.insert(0, str(ROOT))
import importlib
import src.visualization.style as chart_style
importlib.reload(chart_style)
from src.visualization.style import apply_style, display_es, es, format_number, format_percent
apply_style()

REBUILD_PIPELINE = False
HIGH_DEMAND_QUANTILE = 0.75
processed_dir = ROOT / "data/processed"
current_pointer = processed_dir / "CURRENT.json"
if REBUILD_PIPELINE or not current_pointer.is_file():
    subprocess.run([
        sys.executable, str(ROOT / "src/data/process_data.py"),
        "--high-demand-quantile", str(HIGH_DEMAND_QUANTILE),
    ], cwd=ROOT, check=True)
print("Current publication pointer:", current_pointer)

Current publication pointer: /Users/nataliaburguillomartin/PDRL & MLLB/fashion-demand-ml/data/processed/CURRENT.json


## 2. Load the current validated run

The publication pointer is resolved inside `data/processed`, previews are rejected, and every consumed table is checked against its recorded SHA-256 digest. Identifiers retain leading zeros and week columns are parsed as dates.

**Code walkthrough.** The next cell reads `CURRENT.json`, checks that the published files match their recorded checksums, and loads the article-week and clustering tables.


In [2]:
pointer = json.loads(current_pointer.read_text(encoding="utf-8"))
run_dir = (processed_dir / pointer["run_dir"]).resolve()
if not run_dir.is_relative_to(processed_dir.resolve()) or pointer.get("preview"):
    raise ValueError("CURRENT.json does not reference a valid complete publication.")

summary = json.loads((run_dir / "summary.json").read_text(encoding="utf-8"))
provenance = json.loads((run_dir / "provenance.json").read_text(encoding="utf-8"))
schema = json.loads((run_dir / "schema.json").read_text(encoding="utf-8"))
checksums = json.loads((run_dir / "checksums.json").read_text(encoding="utf-8"))

def verify_checksum(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(4 * 1024 * 1024), b""):
            digest.update(block)
    if digest.hexdigest() != checksums[path.name]:
        raise ValueError(f"{path.name} changed after publication.")

article_week_path = run_dir / "article_week.csv.gz"
clustering_path = run_dir / "article_clustering.csv.gz"
for published_path in [article_week_path, clustering_path]:
    verify_checksum(published_path)

model_data = pd.read_csv(
    article_week_path,
    dtype={"article_id": "string", "product_code": "string"},
    parse_dates=["week", "target_week"],
)
clustering_data = pd.read_csv(
    clustering_path,
    dtype={"article_id": "string", "product_code": "string"},
    parse_dates=["observation_start", "observation_end"],
)
print("Run:", run_dir.name)
display_es(pd.DataFrame({
    "rows": {name: details["rows"] for name, details in schema.items()},
    "columns": {name: len(details["dtypes"]) for name, details in schema.items()},
}))
display_es(model_data.head())

Run: 20261024T182146-1559e91a


,rows,columns
article_week,1.887.139,51
articles_clean,105.542,25
article_clustering,77.357,24


,week,article_id,sales,min_price,max_price,online_sales,mean_price,online_share,lag_1_sales,lag_2_sales,lag_3_sales,lag_4_sales,lag_5_sales,lag_6_sales,lag_7_sales,lag_8_sales,lag_9_sales,lag_10_sales,lag_11_sales,lag_12_sales,target_next_week_sales,target_week,target_is_observed,history_is_observed,rolling_4w_sales_mean,rolling_4w_sales_std,rolling_4w_active_weeks,rolling_12w_sales_mean,rolling_12w_sales_std,rolling_12w_active_weeks,sales_trend_4w,lag_1_mean_price,lag_2_mean_price,lag_3_mean_price,lag_4_mean_price,rolling_4w_mean_price,price_change_from_4w_mean,rolling_4w_online_share,iso_year,iso_week,split,product_code,product_type_name,product_group_name,graphical_appearance_name,colour_group_name,perceived_colour_value_name,index_group_name,section_name,garment_group_name,target_high_demand
0,2018-12-17 00:00:00,0108775015,227,"0,0063389830508474","0,008457627118644",178,"0,0083214365713431","0,7841409691629956",300,337,298,680,244,312,373,434,259,360,315,547,229,2018-12-24 00:00:00,True,True,"403,75","160,2472698675394",4,"371,5833333333333","121,57402294715575",12,"-170,5","0,008290056497175","0,0081476135392042","0,0083140143328403","0,0072565054835492","0,0080020474631922","0,0003193891081509","0,7715170278637771",2.018,51,train,0108775,Vest top,Garment Upper body,Solid,Black,Dark,Ladieswear,Womens Everyday Basics,Jersey Basic,1
1,2018-12-17 00:00:00,0108775044,107,"0,0030338983050847","0,008457627118644",72,"0,0082326944400442","0,6728971962616822",108,106,97,240,81,92,107,160,115,155,139,226,107,2018-12-24 00:00:00,True,True,"137,75","59,17928269250989",4,"135,5","49,4191258522447",12,"-61,5","0,0082432517263025","0,0081325551646945","0,0083047352786999","0,0073318502824858","0,0080030981130456","0,0002295963269985","0,6987295825771325",2.018,51,train,0108775,Vest top,Garment Upper body,Solid,White,Light,Ladieswear,Womens Everyday Basics,Jersey Basic,1
2,2018-12-17 00:00:00,0108775051,1,"0,0050677966101694","0,0050677966101694",1,"0,0050677966101694",1,2,3,1,4,2,4,5,19,24,44,45,38,0,2018-12-24 00:00:00,True,True,"2,5","1,118033988749895",4,"15,916666666666666","16,77527115197314",12,0,"0,0045593220338982","0,0047288135593219","0,0050677966101694","0,0045593220338982","0,0047288135593219","0,0003389830508474",1,2.018,51,train,0108775,Vest top,Garment Upper body,Stripe,Off White,Dusty Light,Ladieswear,Womens Everyday Basics,Jersey Basic,0
3,2018-12-17 00:00:00,0110065001,18,"0,021593220338983","0,0254067796610169",7,"0,0249133709981167","0,3888888888888889",14,31,14,64,25,30,25,50,23,45,37,56,22,2018-12-24 00:00:00,True,True,"30,75","20,41292482717751",4,"34,5","15,47847968417226",12,"-16,5","0,0244104116222759","0,0231667577911426","0,0244987893462469","0,0218728813559321","0,0234872100288994","0,0014261609692172","0,6016260162601627",2.018,51,train,0110065,Bra,Underwear,Solid,Black,Dark,Ladieswear,Womens Lingerie,"Under-, Nightwear",1
4,2018-12-17 00:00:00,0110065002,21,"0,0169322033898305","0,0254067796610169",18,"0,0193736884584341","0,8571428571428571",6,5,5,23,10,8,17,16,12,20,16,16,31,2018-12-24 00:00:00,True,True,"9,75","7,660776723022281",4,"12,833333333333334","5,799904213768438",12,"-8,5","0,0245593220338982","0,0228644067796609","0,0233728813559321","0,0227546057479734","0,0233878039793661","-0,0040141155209319","0,8974358974358975",2.018,51,train,0110065,Bra,Underwear,Solid,White,Light,Ladieswear,Womens Lingerie,"Under-, Nightwear",1


## 3. Audit the processed article-week table

The checks inspect the complete table without silently dropping rows. Historical price and channel features may be missing when an article had no recorded sale in the relevant prior weeks; those missing values must later be imputed inside each training pipeline. Other missing or infinite values are treated as data-quality failures.

**Code walkthrough.** The next cell checks the full table for duplicate article-weeks, unexpected missing values, invalid values, and the expected sales period.


In [3]:
allowed_missing_columns = {
    "lag_1_mean_price", "lag_2_mean_price", "lag_3_mean_price", "lag_4_mean_price",
    "rolling_4w_mean_price", "price_change_from_4w_mean", "rolling_4w_online_share",
}
missing_by_column = model_data.isna().sum()
unexpected_missing = missing_by_column.drop(labels=list(allowed_missing_columns), errors="ignore").sum()
numeric_values = model_data.select_dtypes(include="number")
tolerance = 1e-12
issues = pd.Series({
    "Duplicate article-week rows": model_data.duplicated(["article_id", "week"]).sum(),
    "Unexpected missing values": unexpected_missing,
    "Infinite numeric values": np.isinf(numeric_values.to_numpy()).sum(),
    "Negative sales values": (model_data[["sales", "target_next_week_sales"]] < 0).any(axis=1).sum(),
    "Negative price values": (model_data[["min_price", "mean_price", "max_price"]] < 0).any(axis=1).sum(),
    "Mean price outside min-max range": (
        (model_data["mean_price"] < model_data["min_price"] - tolerance)
        | (model_data["mean_price"] > model_data["max_price"] + tolerance)
    ).sum(),
    "Invalid online share": (~model_data["online_share"].between(0, 1)).sum(),
    "Online sales above total sales": (model_data["online_sales"] > model_data["sales"]).sum(),
    "Unobserved targets": (~model_data["target_is_observed"]).sum(),
    "Unobserved history": (~model_data["history_is_observed"]).sum(),
}, dtype="int64")
display_es(missing_by_column[missing_by_column.gt(0)].sort_values(ascending=False).to_frame("missing"))
display_es(issues.to_frame("count"))
if issues.any():
    raise ValueError("Resolve the reported data-quality issues before continuing.")
display_es(model_data[["sales", "mean_price", "target_next_week_sales"]].describe())

,missing
lag_4_mean_price,529.782
lag_3_mean_price,490.922
lag_2_mean_price,450.695
lag_1_mean_price,409.376
rolling_4w_mean_price,162.010
price_change_from_4w_mean,162.010
rolling_4w_online_share,162.010


,count
Duplicate article-week rows,0
Unexpected missing values,0
Infinite numeric values,0
Negative sales values,0
Negative price values,0
Mean price outside min-max range,0
Invalid online share,0
Online sales above total sales,0
Unobserved targets,0
Unobserved history,0


,sales,mean_price,target_next_week_sales
count,1.887.139,1.887.139,1.887.139
mean,"14,677076781307576","0,028212171990686312","13,830034777512415"
std,"35,89778156792835","0,02333421772211515","35,066464572946224"
min,1,"0,0001016949152542",0
25%,1,"0,0135423728813559",1
50%,4,"0,022864406779661",3
75%,12,"0,0338813559322033",12
max,3.100,"0,5067796610169492",3.100


## 4. Feature engineering and temporal semantics

Every retained row has twelve complete historical weeks, a complete current week, and a complete target week. Sales lags use zero only inside verified coverage. Historical price and channel features remain missing when no price or channel observation exists. A positive `sales_trend_4w` means that the two most recent prior weeks exceed the two earlier weeks on average.

**Code walkthrough.** The next cell groups related columns and verifies that past-sales features refer only to weeks already known when the prediction is made.


In [4]:
feature_groups = {
    "current_week": ["sales", "mean_price", "min_price", "max_price", "online_share"],
    "sales_lags": [f"lag_{lag}_sales" for lag in range(1, 13)],
    "rolling_demand": [column for column in model_data if column.startswith("rolling_") and "price" not in column and "online" not in column],
    "trend": ["sales_trend_4w"],
    "historical_price": [column for column in model_data if "mean_price" in column and column != "mean_price"] + ["price_change_from_4w_mean"],
    "historical_channel": ["rolling_4w_online_share"],
    "calendar": ["iso_year", "iso_week"],
    "product_metadata": ["product_type_name", "product_group_name", "graphical_appearance_name",
                         "colour_group_name", "perceived_colour_value_name", "index_group_name",
                         "section_name", "garment_group_name"],
}
display_es(pd.Series({name: len(columns) for name, columns in feature_groups.items()}, name="feature_count").to_frame())

assert (model_data["target_week"] == model_data["week"] + pd.Timedelta(weeks=1)).all()
iso_calendar = model_data["week"].dt.isocalendar()
assert (model_data["iso_year"] == iso_calendar.year).all()
assert (model_data["iso_week"] == iso_calendar.week).all()
previous_rows = model_data[["article_id", "week", "sales"]].copy()
previous_rows["week"] += pd.Timedelta(weeks=1)
previous_rows = previous_rows.rename(columns={"sales": "previous_observed_sales"})
matched_lag = model_data[["article_id", "week", "lag_1_sales"]].merge(
    previous_rows, on=["article_id", "week"], how="inner", validate="one_to_one")
assert matched_lag["lag_1_sales"].eq(matched_lag["previous_observed_sales"]).all()
print("Verified lag-1 values against", f"{es(len(matched_lag)):,}", "consecutive article-week rows.")

,feature_count
current_week,5
sales_lags,12
rolling_demand,6
trend,1
historical_price,6
historical_channel,1
calendar,2
product_metadata,8


Verified lag-1 values against 1.459.139 consecutive article-week rows.


## 5. Audit chronological partitions

The pipeline approximates the requested 70/15/15 row allocation without splitting target weeks. Boundaries use row counts rather than target values. Optional gap weeks are reported separately and excluded from percentage denominators.

**Code walkthrough.** The next cell checks that training, validation, and test contain complete weeks in chronological order and reports the actual row percentages.


In [5]:
split_config = summary["split_configuration"]
split_order = ["train", "validation", "test"]
split_summary = model_data.groupby("split").agg(
    rows=("article_id", "size"),
    articles=("article_id", "nunique"),
    target_weeks=("target_week", "nunique"),
    first_target=("target_week", "min"),
    last_target=("target_week", "max"),
).reindex(split_order)
if split_summary.isna().any().any():
    raise ValueError("At least one chronological partition is empty.")
split_summary["actual_percentage"] = (100 * split_summary["rows"] / split_summary["rows"].sum()).round(2)
split_summary["requested_percentage"] = pd.Series(split_config["requested_percentages"])
display_es(split_summary)
for earlier, later in [("train", "validation"), ("validation", "test")]:
    assert split_summary.loc[earlier, "last_target"] < split_summary.loc[later, "first_target"]
assert all(model_data.groupby("target_week")["split"].nunique().eq(1))

,rows,articles,target_weeks,first_target,last_target,actual_percentage,requested_percentage
split,,,,,,,
train,1.326.434,77.357,62,2018-12-24 00:00:00,2020-02-24 00:00:00,"70,29",70
validation,271.215,40.820,15,2020-03-02 00:00:00,2020-06-08 00:00:00,"14,37",15
test,289.490,43.869,14,2020-06-15 00:00:00,2020-09-14 00:00:00,"15,34",15


## 6. Classification target

High demand is defined using the configured quantile of **positive next-week sales in training only**. The observed integer at or above that quantile becomes a fixed unit threshold and is applied unchanged to validation and test. This class-definition threshold is distinct from the probability decision threshold later chosen for a classifier.

**Code walkthrough.** The next cell recalculates the high-demand rule using training rows, then checks that the same threshold labels validation and test.


In [6]:
classification = summary["classification_target"]
threshold = classification["threshold_units"]
expected_class = model_data["target_next_week_sales"].ge(threshold).astype("int8")
assert model_data["target_high_demand"].isin([0, 1]).all()
assert model_data["target_high_demand"].eq(expected_class).all()
train_positive = model_data.loc[
    model_data["split"].eq("train") & model_data["target_next_week_sales"].gt(0),
    "target_next_week_sales",
]
recomputed_threshold = int(max(1, train_positive.quantile(
    classification["training_quantile"], interpolation="higher")))
assert threshold == recomputed_threshold
class_summary = model_data.groupby("split").agg(
    rows=("target_high_demand", "size"),
    high_demand_rows=("target_high_demand", "sum"),
    prevalence=("target_high_demand", "mean"),
).reindex(split_order)
display_es(pd.Series(classification, name="value").to_frame())
# Format only the displayed copy; keep numeric prevalence for later checks.
class_summary_display = class_summary.copy()
class_summary_display["prevalence"] = class_summary_display["prevalence"].map(
    lambda value: f"{es(value):.2%}"
)
display_es(class_summary_display)

,value
definition,target_next_week_sales >= threshold
threshold_units,16
training_quantile,"0,75"
reference_population,positive target_next_week_sales rows in train only
prevalence_by_split,"{'test': 0.21140972054302393, 'train': 0.1992432341149277, 'validation': 0.21316667588444593}"


,rows,high_demand_rows,prevalence
split,,,
train,1.326.434,264.283,"19,92 %"
validation,271.215,57.814,"21,32 %"
test,289.490,61.201,"21,14 %"


## 7. Leakage-safe clustering representation

The clustering table contains one row per article and uses only current and historical signals from the training period. It includes demand level, volatility, trend, activity, price, channel mix, and product metadata. Future regression and classification targets are not used. Scaling, encoding, dimensionality reduction, and cluster selection remain the responsibility of `09_unsupervised.ipynb`.

**Code walkthrough.** The next cell checks the separate article-level table and confirms that it does not contain future targets.


In [7]:
target_like_columns = [column for column in clustering_data if column.startswith("target_")]
assert not target_like_columns
assert not clustering_data["article_id"].duplicated().any()
assert clustering_data["observation_end"].le(
    model_data.loc[model_data["split"].eq("train"), "week"].max()).all()
clustering_summary = pd.Series({
    "articles": len(clustering_data),
    "columns": clustering_data.shape[1],
    "numeric_descriptors": clustering_data.select_dtypes(include="number").shape[1],
    "categorical_descriptors": clustering_data.select_dtypes(exclude=["number", "datetime"]).shape[1] - 1,
    "future_target_columns": len(target_like_columns),
})
display_es(clustering_summary.to_frame("value"))
display_es(clustering_data.head())

,value
articles,77.357
columns,24
numeric_descriptors,12
categorical_descriptors,9
future_target_columns,0


,article_id,observation_start,observation_end,article_weeks,mean_current_week_sales,median_current_week_sales,std_current_week_sales,max_current_week_sales,mean_12w_sales,mean_12w_sales_volatility,mean_sales_trend_4w,mean_price,price_std,historical_active_share,mean_online_share,product_code,product_type_name,product_group_name,graphical_appearance_name,colour_group_name,perceived_colour_value_name,index_group_name,section_name,garment_group_name
0,0108775015,2018-12-17 00:00:00,2020-02-17 00:00:00,58,"107,67241379310344","12,5","135,691133782034",382,"147,45545977011494","50,30440385053417","-13,577586206896552","0,0077511060370717","0,0011789796020489","0,9497126436781608","0,8901521216973579",0108775,Vest top,Garment Upper body,Solid,Black,Dark,Ladieswear,Womens Everyday Basics,Jersey Basic
1,0108775044,2018-12-17 00:00:00,2020-02-17 00:00:00,62,"85,01612903225806",20,"95,94029768140282",312,"97,47849462365592","35,80848791824991","-4,233870967741935","0,0079273247395035","0,0006861034475723",1,"0,7406564219313223",0108775,Vest top,Garment Upper body,Solid,White,Light,Ladieswear,Womens Everyday Basics,Jersey Basic
2,0108775051,2018-12-17 00:00:00,2019-06-24 00:00:00,3,1,1,0,1,"5,333333333333333","5,683885517056474",0,"0,0073276836158191","0,001957119556575","0,3611111111111111","0,6666666666666666",0108775,Vest top,Garment Upper body,Stripe,Off White,Dusty Light,Ladieswear,Womens Everyday Basics,Jersey Basic
3,0110065001,2018-12-17 00:00:00,2020-02-17 00:00:00,62,"9,661290322580646",8,"6,301036990863702",28,"12,313172043010752","6,222794391879432","-0,8629032258064516","0,0174007978129184","0,0062229319013287",1,"0,2821368948247078",0110065,Bra,Underwear,Solid,Black,Dark,Ladieswear,Womens Lingerie,"Under-, Nightwear"
4,0110065002,2018-12-17 00:00:00,2020-02-17 00:00:00,54,"6,648148148148148",3,"9,761606362176815",46,"7,308641975308642","5,1988837492626585","-0,1851851851851851","0,0176741381331918","0,0059968573035428","0,8765432098765432","0,6740947075208914",0110065,Bra,Underwear,Solid,White,Light,Ladieswear,Womens Lingerie,"Under-, Nightwear"


## 8. Leakage contract and published outputs

Preprocessing parameters, category encodings, scaling, feature selection, class weights, and decision thresholds must be learned inside the permitted training window. Do not use `target_next_week_sales`, `target_high_demand`, `target_week`, `split`, `target_is_observed`, or `history_is_observed` as predictors. Identifiers remain available for joins and auditing, not as continuous quantities. At publication time, the final test period is reserved from model fitting and selection. Notebook 07 has since evaluated that period, including an earlier sampled run. Its current full-period test scores are therefore exploratory, not a fresh untouched holdout estimate. Any future confirmatory evaluation requires new unseen data.

The versioned publication contains:

- `article_week.csv.gz`: supervised article-week table and chronological splits.
- `article_clustering.csv.gz`: training-history article descriptors without future targets.
- `articles_clean.csv.gz`: validated product metadata.
- `summary.json`, `provenance.json`, `schema.json`, and `checksums.json`: audit and reproducibility metadata.

## Conclusion

The resulting datasets share a documented temporal contract across regression, classification, and clustering. Historical transformations and the class definition are derived without consulting future partitions, while optional missing historical price or channel observations are left for pipeline-based imputation during model fitting.